# Silent Speech Interpretability

## Professor-facing results walkthrough

This notebook summarizes the tracked evidence for contactless / microphone-free speech decoding. Audio is used only to construct teacher targets; student inference uses non-audio sensors.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

TABLES = ROOT / "reports" / "tables"
required = [
    TABLES / "project_key_results.csv",
    TABLES / "external_radar_generalization_summary.csv",
    TABLES / "external_radar_feature_ablation.csv",
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing tracked report tables: {missing}")

pd.set_option("display.max_colwidth", 80)

## 1. Project map

The project has two central tracks:

1. **Strict supervised baseline:** 30-class RVTALL decoding with speaker- and encoder-disjoint evaluation.
2. **Audio-teacher interpretability:** transfer HuBERT speech structure into silent-sensor students, then probe temporal, articulatory, phonetic, sparse, and causal structure.

External radar evaluation tests whether the resulting method survives a new laboratory, language, sensor, and cohort.

In [ ]:
key_results = pd.read_csv(TABLES / "project_key_results.csv")
presentation = key_results[["track", "evaluation", "metric", "value", "chance", "status"]].copy()
presentation["value"] = presentation["value"].map(lambda value: f"{value:.3f}")
presentation["chance"] = presentation["chance"].map(lambda value: "" if pd.isna(value) else f"{value:.3f}")
display(presentation)

## 2. External generalization boundary

![External radar generalization](../reports/figures/final_external_generalization.svg)

The independent radar student transfers across recording sessions with familiar subjects, but collapses under leave-one-subject-out evaluation. This is the main unresolved generalization boundary.

In [ ]:
external = pd.read_csv(TABLES / "external_radar_generalization_summary.csv")
metrics = ["accuracy", "segment_cosine", "order_margin_reversed", "residual_order_margin_reversed"]
external_view = external[external["metric"].isin(metrics)].copy()
external_view = external_view[[
    "protocol", "metric", "mean", "unit_bootstrap_95_low",
    "unit_bootstrap_95_high", "mean_within_unit_seed_sd"
]]
display(external_view.round(3))

## 3. What the external radar student uses

![External radar feature ablations](../reports/figures/final_external_feature_ablation.svg)

S32 and magnitude structure carry most standalone performance. Temporal differences and S12 are weaker alone, although the combined representation gives the strongest HuBERT alignment.

In [ ]:
ablations = pd.read_csv(TABLES / "external_radar_feature_ablation.csv")
ablation_summary = (
    ablations.groupby("feature_set")
    .agg(
        accuracy=("accuracy", "mean"),
        hubert_cosine=("segment_cosine", "mean"),
        order_margin=("order_margin_reversed", "mean"),
    )
    .sort_values("accuracy", ascending=False)
)
display(ablation_summary.round(3))

## 4. Interpretation boundary

### Supported

- 63.9% strict encoder-disjoint RVTALL fusion accuracy.
- Useful class and coarse ordered HuBERT information in silent-sensor students.
- Modest broad-phone occupancy information beyond class and relative position.
- Independent external replication across radar recording sessions.

### Not supported

- Broad external radar generalization to unseen speakers.
- Exact phone tracking from four relative-time segments.
- Stable one-feature/one-phoneme selectivity.
- An assumption that adding attention automatically improves interpretability.

The full narrative and evidence map are in [`reports/final_project_report.md`](../reports/final_project_report.md).

## 5. Reproducibility

```bash
python3 -m pip install -e '.[audio-teachers,interpretability,alignment]'
make test
make external-radar-validation-batch
make final-package
```

Large corpora, embeddings, teacher targets, activations, and checkpoints remain ignored. The notebook, compact tables, reports, and static figures are tracked.